In [ ]:
import os
import numpy as np
import pandas as pd
import shutil
import cv2
import random
from PIL import Image
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import warnings

warnings.filterwarnings("ignore")

# === CONFIG ===
IMG_SIZE = 224
FEATURE_SIZE = 243  # For 9x9x3 image
IMG_SHAPE = (9, 9, 3)
TRAIN_DIR = './Images_smot/'
RESIZED_TRAIN_DIR = './Images_resized_smot/'

In [1]:
import pandas as pd
import numpy as np
import os
from PIL import Image
from sklearn.preprocessing import MinMaxScaler
from imblearn.over_sampling import SMOTE

# === 1. Load & Clean Dataset ===
df = pd.read_csv('train_data.csv', low_memory=False)
df.columns = df.columns.str.strip()
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

# === 2. Label Mapping ===
def map_label(label):
    if label == "BENIGN": return "BENIGN"
    elif "DoS" in label: return "DoS"
    elif "PortScan" in label or "Brute Force" in label: return "Fuzzy"
    elif "Web Attack" in label or "Sql Injection" in label or "XSS" in label: return "RPM"
    elif "Bot" in label or "Infiltration" in label or "Heartbleed" in label: return "Advanced"
    elif "Patator" in label: return "Patator"
    else: return "Other"

# Apply the mapping to the dataframe
df['Label'] = df['Label'].apply(map_label)

# Separate features (X) and target (y)
X = df.drop('Label', axis=1)
y = df['Label']



In [2]:
from collections import Counter
print("Class distribution BEFORE SMOTE:")
print(Counter(y))


Class distribution BEFORE SMOTE:
Counter({'BENIGN': 1362791, 'DoS': 227842, 'Fuzzy': 96186, 'Patator': 8299, 'Advanced': 1203, 'RPM': 404})


In [3]:
# Record the length of the original data BEFORE SMOTE
original_length = len(X)
TARGET_SAMPLES = 6000 
Patator_TARGET_SAMPLES = 20000 

current_counts = Counter(y)

# Create a dictionary for the sampling strategy.
# The max() function ensures we don't accidentally downsample or throw an error 
# if a class already has more samples than TARGET_SAMPLES.
strategy = {
    'RPM': max(current_counts.get('RPM', 0), TARGET_SAMPLES),
    'Advanced': max(current_counts.get('Advanced', 0), TARGET_SAMPLES)
    
}
# === 1. Apply SMOTE ===
# (Assuming 'strategy' dictionary is already defined as in the previous step)
smote = SMOTE(sampling_strategy=strategy, random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

# === 2. Isolate ONLY the Generated Data ===
# Slice the arrays starting from the end of the original data
X_aug = X_resampled.iloc[original_length:]
y_aug = y_resampled.iloc[original_length:]

print(f"Original samples: {original_length}")
print(f"Generated samples: {len(X_aug)}")

# === 3. Feature Scaling ===
# Fit the scaler on the ORIGINAL data, but transform the AUGMENTED data
scaler = MinMaxScaler()
scaler.fit(X) 
X_aug_scaled = scaler.transform(X_aug)

# Convert to 0-255 for image pixels
X_aug_scaled = (X_aug_scaled * 255).astype(np.uint8)

# === 4. Create Padded Images ===
FEATURE_SIZE = 81 # Example: 9x9 = 81. Adjust to your needs.
IMG_SHAPE = (9, 9)

def create_image_from_row(row, size=FEATURE_SIZE):
    data = np.array(row)
    if len(data) < size:
        data = np.pad(data, (0, size - len(data)), 'constant')
    else:
        data = data[:size]
    return data.reshape(IMG_SHAPE).astype(np.uint8)

# === 5. Save Images by Class (with Prefix) ===
# Added a 'prefix' argument to handle the "aug_" naming requirement
def save_images_by_class(X_data, y_data, base_path, prefix=""):
    for idx, (row, label) in enumerate(zip(X_data, y_data)):
        img = create_image_from_row(row)
        class_dir = os.path.join(base_path, label)
        os.makedirs(class_dir, exist_ok=True)
        
        # Save image as aug_0.png, aug_1.png, etc.
        filename = f"{prefix}{idx}.png"
        Image.fromarray(img).save(os.path.join(class_dir, filename))

# Run the saving function ONLY on the augmented data
TRAIN_DIR = 'Images/train/'
save_images_by_class(X_aug_scaled, y_aug, TRAIN_DIR, prefix="aug_")

print("Augmented images saved successfully with 'aug_' prefix.")

Original samples: 1696725
Generated samples: 10393
Augmented images saved successfully with 'aug_' prefix.


In [2]:
import os

train_dir = 'Images/train'
# Define the image extensions you want to look for
valid_extensions = ('.png', '.jpg', '.jpeg') 

total_images = 0

print("--- Count per subfolder ---")
for folder_name in os.listdir(train_dir):
    folder_path = os.path.join(train_dir, folder_name)
    
    # Check if it's actually a directory
    if os.path.isdir(folder_path):
        # Count only files that end with your valid image extensions
        images = [f for f in os.listdir(folder_path) if f.lower().endswith(valid_extensions)]
        count = len(images)
        print(f"{folder_name}: {count} images")
        total_images += count

print("-" * 27)
print(f"Total images: {total_images}")

--- Count per subfolder ---
DoS: 227842 images
Fuzzy: 96186 images
Patator: 8299 images
RPM: 6000 images
BENIGN: 1362791 images
Advanced: 6000 images
---------------------------
Total images: 1707118


In [ ]:


# === Load & Clean Dataset ===
df = pd.read_csv('CICIDS2017_balanced.csv', low_memory=False)
df.columns = df.columns.str.strip()
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)
# === Label Mapping ===
def map_label(label):
    if label == "BENIGN": return "BENIGN"
    elif "DoS" in label: return "DoS"
    elif "PortScan" in label or "Brute Force" in label: return "Fuzzy"
    elif "Web Attack" in label or "Sql Injection" in label or "XSS" in label: return "RPM"
    elif "Bot" in label or "Infiltration" in label or "Heartbleed" in label: return "Advanced"
    elif "Patator" in label: return "Patator"
    else: return "Other"

# Apply mapping
df['Category'] = df['Label'].apply(map_label)
df = df[df['Category'] != "Other"]

# === Feature Scaling ===
features = df.select_dtypes(include=[np.number])
labels = df['Category']
scaler = MinMaxScaler()
features_scaled = scaler.fit_transform(features)
features_scaled = (features_scaled * 255).astype(np.uint8)

# === Create Padded Images ===
def create_image_from_row(row, size=FEATURE_SIZE):
    data = np.array(row)
    if len(data) < size:
        data = np.pad(data, (0, size - len(data)), 'constant')
    else:
        data = data[:size]
    return data.reshape(IMG_SHAPE).astype(np.uint8)

# === Save Images by Class ===
def save_images_by_class(X, y, base_path):
    for idx, (row, label) in enumerate(zip(X, y)):
        img = create_image_from_row(row)
        class_dir = os.path.join(base_path, label)
        os.makedirs(class_dir, exist_ok=True)
        Image.fromarray(img).save(os.path.join(class_dir, f"{idx}.png"))


save_images_by_class(features_scaled, labels, TRAIN_DIR)

print("Images saved by class in train/test folders.")

# === Resize to 224x224 ===


In [ ]:
def resize_images(src_dir, dst_dir):
    for root, dirs, files in os.walk(src_dir):
        for name in files:
            src_path = os.path.join(root, name)
            img = cv2.imread(src_path)
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            dst_path = root.replace(src_dir, dst_dir)
            os.makedirs(dst_path, exist_ok=True)
            cv2.imwrite(os.path.join(dst_path, name), img)
    print(f"Resized images saved to: {dst_dir}")

resize_images(TRAIN_DIR, RESIZED_TRAIN_DIR)

print("All done.")
